In [1]:
from transformers import pipeline # pipeline act as the API to get the model from the hugging face
                                  # hugging face is known as the hub for getting pretrained machine learning models. Similar to GitHub repository.
import pandas as pd
from sklearn.linear_model import LinearRegression
import torch # torch is the engine behind many modern AI technologies. It can move massive calculations from your computer's CPU to its GPU to save time. Many other functions are also there
import gradio as gr # gradio is used to create the User  Interface for our AI Data Analyst

In [2]:
classifier=pipeline("zero-shot-classification",model= "facebook/bart-large-mnli") # pip install huggingface_hub[hf_xet] - The installation of this library helps to improve the speed of retrieving only zero-shot-classification from faceboko/bart model present in hugging face repository
                                 # here we assign the zero-shot-classification's function from facebook model to 'classifier' variable. so we can use classifier variable to perform this function of the  facebook model's package

Device set to use cpu


In [3]:
# zero-shot-classification : Zero-shot-classification is a powerful Artificial Intelligence technique that allows a model to categorize text into labels it has never been specifically trained on before.
# The model takes your input (the prompt) and compares it against a list of candidate_labels you provided: ["descriptive_statistics", "correlation", "regression", "others"]. Its job is to determine which of these tasks you are asking for.

In [4]:
def classify_prompt(prompt):  # we are defining a function here named 'classify_prompt'. So that we can use this function anywhere in this code by just calling by this name
    labels= ["descriptive_statistics","correlation","regression","others"] # Assigning a list of names to 'labels' variable.
    result=classifier(prompt,candidate_labels=labels) # here classifier is the function we made using the facebook model and we are giving parameters inside this function to perform the funtion we want. #'candidate_labels': You are giving the AI a specific list of categories to choose from: ["descriptive_statistics", "correlation", "regression", "others"].
    return result["labels"][0] # we will get 4 labels from the prompt as result, but we want only the label with highest confident value, that means: the label with highest similarity with the word in our prompt and it automatically comes to the first pos in a list

In [5]:
import spacy # It breaks text into individual words and punctuation mark


In [6]:
nlp = spacy.load('en_core_web_sm') # the package 'en_core_web_sm' in spacy library identifies the part of speech and label variables in sentence to entities like organization,location, money etc
                                   # and here we assign the function of that package to nlp. so we can use nlp to perform this function of the package

In [7]:
def variable_extract(prompt, columns):  # definig a new variable to extract the variables in our prompt
    doc = nlp(prompt)
    
    # Method 1: spaCy Entities
    allowed_labels = ["GPE", "ORG", "PRODUCT", "LOC", "PERSON"] # these are the posssible entites that can appear in our prompt and we are assisgning the name of that entites toa list named 'allowed_labels'
    found_vars = [ent.text for ent in doc.ents if ent.label_ in allowed_labels] # in 'found_vars' list, a for loop takes place with iteration over each words in the prompt one by one and checks the similarity with the labels in the 'allowed_labels' list and those matched words are added to the 'found_vars' list.
                                                                                # we need only text entities and that's why ent.text is written in the code
                                                                                # doc.ents contains all the words in the prompt
    # Method 2: Phrase Matching (Fixes the "Area Income" issue)
    # 1. Clean punctuation but do NOT split into individual words
    cleaned_prompt = prompt.replace(',', ' ').replace('.', ' ')  # it replaces all the commas and full stops from the prompt with spaces but doesn't extract words one by one. it remains as a long text
    
    # 2. Loop through the actual column names from your CSV
    for col in columns:
        # 3. Check if the full column name (e.g., "Area Income") is in the prompt
        # We also check "col not in found_vars" to avoid duplicates
        if col in cleaned_prompt and col not in found_vars: #if we have a two word named column and it will surely be splitted in the found_vars list but it will be present as a part of the sentence in the cleaned_prompt and if we type that name in the prompt, surely that name will be extarcted as one. # here each column names is checked with each words in the prompt # this means if the names in cleaned_propmt list matches with column names (eg: column name with spaces: "Area Income" and doesn't match with all the words in the prompt which is found in found_vars list
            found_vars.append(col) # If the column name was found in your prompt and isn't already in our 'found_vars' list, this line adds it to the 'found_vars' list.
                
    return found_vars if found_vars else None  # 'if found_vars' command  means that the list 'found_vars' is not empty and then that list will be returned or displayed . 'else None' command means list 'found_vars' is empty and it returns nothing.

In [8]:
def load_data(file): # this user degined function helps to load only a csv file
    if not file.name.endswith(".csv"):
        raise valueError("Please Upload a CSV File") # 'raise' : This is a command that tells Python to stop whatever it is doing immediately because a specific condition was violated.
                                                    # ValueError: This is a specific type of "Exception" in Python used when a function receives an input that has the right type (a file) but an inappropriate "value" (the wrong format).
    return pd.read_csv(file.name)

In [9]:
def get_columnns(df): # this user defined function helps to retrieve the column names in a table and displays it in a list
    return df.columns.tolist()

In [10]:
def descriptive_statistics(df, variables):  # here the inputs are data frame and variables that are typed in the prompt and that variables are used for analysis with the provided data frame in the file
    return df[variables].describe()  # describe() imbulit function gives mean, median,sd etc.

In [11]:
def correlation(df,variables):
    return df[variables].corr() # this will give the relation between the variables u enterd in the prompt by taking these variabl column from the dataframe in the file provided

In [12]:
def regression(df, dependent_var, independent_vars): # dependent_vars is also known as target variable
    x= df[independent_vars]
    y= df[dependent_var]
    model=LinearRegression() # the LinearRegression() function is assigned to 'model' variable
    model.fit(x,y) # 'fit' tells the model to look at the data in x and y and find the best-fitting straight line that connects them.
                   # During this process, the model calculates the specific coefficients (slopes) and the intercept 
    return model.coef_,model.intercept_  # this function takes the independent variables and target variabel and gives out the coefficent and intercept

In [13]:
def analyze_data(file, prompt): # this user defined function is the brain of our AI Data Analysis Assistant
    # 1. Classify the type of analysis requested
    analysis_type = classify_prompt(prompt) # using the classify_prompt() user defined function, we are classifying the required statistics we want to do after getting the pr ompt
    
    # 2. Load the data first to get the available columns
    try:
        df = load_data(file)
    except Exception as e:  #This catches almost any kind of error (e.g., the file is empty, it's not actually a CSV, or the file name is missing). # if any error occurs, the error command is stored in 'e'. it is like a brief case which contains all the error commands which is not understandable to the normanl users
        return f"Error: Please upload a valid CSV file. {str(e)}"   # this command return the error command in a readable string format to the user. so thta we use str() function here to convert all the commands inside 'e' 
                                                                   # Output be like- Error: Please upload a valid CSV file. [Technical details about why the file failed]

    columns = get_columnns(df)

    # 3. Extract variables using BOTH the prompt and the known column names
    variables = variable_extract(prompt, columns)
    
    if variables is None: 
        return f"Could not find variables in your prompt that matches with the column names in the CSV file. \n Available columns: {', '.join(columns)}"  # ', '.join(...): This takes the list of column names and turns it into a single piece of text separated by commas and spaces.
    
    # 4. Perform the requested analysis
    if set(variables).issubset(columns): # set function is used to collect all the variables from the prompt(variables can be names , locations, etc) and remove duplicates and store, here it means if the variables in the prompt ,match with the column names then move to next step
        if analysis_type == "descriptive_statistics": # then we start the nested if and it states that if the variable in 'analysis_type' matches with ' descriptive statistics' related term from the prompt which is being extracted using 'classify_prompt()' user defined function
            return str(df[variables].describe()) # str(...): This converts the mathematical table created by .describe() into a plain text string. Since your Gradio interface is set to show "text" outputs, this step is necessary to make the table visible in the output box.
        elif analysis_type == "correlation":
            return str(df[variables].corr())
        elif analysis_type == "regression":
            if len(variables) >= 2:
                dependent_var = variables[0]
                independent_vars = variables[1:] # This slice [1:] takes EVERY variable after the first one because target variable or dependent variable can change or affected by more than 1 variable.

                coef, intercept = regression(df, dependent_var, independent_vars) # because of the 'regression()' user defined function, it returns coefficent and intercept and it simultaneously assigns to the 'coef' and 'intercept' lists.

                result_str = f"Regression Results for {dependent_var}:\n"  # Formatting the output to show all coefficients

                for var, c in zip(independent_vars, coef):  # It takes two separate lists—the list of your column names (independent_vars) and the list of mathematical numbers (coef)—and locks them together into pairs.Example: It pairs the name "Area Income" with its specific coefficient number(e.g., 0.005).
                    result_str += f"- {var} Coefficient: {c}\n"  # This line is indented because if you have three variables, this line runs three times to add three different coefficients to your report.
                result_str += f"- Intercept: {intercept}"  # This line is not indented because a regression analysis only has one Intercept.
                return result_str
            else:
                return "Regression requires at least one dependent and one independent variable."
                
                
                         

In [ ]:
interface=gr.Interface(fn=analyze_data, inputs=[gr.File(label="Upload CSV File"), gr.Textbox(label="Analysis Prompt")],outputs= gr.Textbox(label="output", lines=10),title="AI Data Analysis Assistant")
interface.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
